# Geopolitical Escalation and Defence Equity Returns

This notebook examines how abnormal changes in geopolitical conflict intensity — measured from the GDELT Events database — relate to same-day excess returns of defence-sector equities relative to the S&P 500.

**Data sources (both free, no API key required):**
- [GDELT 2.0 Events](https://www.gdeltproject.org/) — global event data updated every 15 minutes
- [yfinance](https://github.com/ranaroussi/yfinance) — market prices via Yahoo Finance

**Methodology:**
1. Download GDELT events and filter for conflict/cooperation events between major geopolitical actors
2. Aggregate daily conflict intensity using Goldstein scale and QuadClass
3. Compute rolling Z-scores to identify abnormal escalation/de-escalation days
4. Event study: measure defence-sector excess returns on shock days
5. Individual stock analysis across core US defence names
6. Market-model abnormal returns and bootstrap significance test
7. Regression-based robustness checks

## 1. Imports and configuration

In [ ]:
import os
import time
import zipfile
import io
import warnings
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 50)

# ── Configuration ──────────────────────────────────────────────────────────────
LOOKBACK_DAYS   = 365          # days of GDELT history to fetch
CACHE_FILE      = 'gdelt_events_cache.csv'
ROLLING_WINDOW  = 30           # days for rolling Z-score baseline
ESC_Z_THRESHOLD = 1.5          # Z-score threshold for escalation spike

# Defence ETF (iShares U.S. Aerospace & Defence) vs broad market
DEFENCE_TICKER  = 'ITA'
MARKET_TICKER   = 'SPY'

# Individual defence names for peer study
DEFENCE_PEERS   = ['LMT', 'RTX', 'NOC', 'GD', 'LHX', 'HWM']

# GDELT QuadClass: 1=Verbal Coop, 2=Material Coop, 3=Verbal Conflict, 4=Material Conflict
CONFLICT_CLASSES    = {3, 4}
COOPERATION_CLASSES = {1, 2}

# Geopolitical actor filter — ISO country codes of major conflict actors
CONFLICT_ACTORS = {
    'RUS', 'UKR', 'ISR', 'PSE', 'IRN', 'CHN', 'PRK',
    'USA', 'GBR', 'FRA', 'DEU', 'SAU', 'SYR', 'YEM',
    'TWN', 'LBN', 'PAK', 'IND'
}

print('Configuration loaded.')

## 2. GDELT data ingestion

GDELT publishes a master file list at `http://data.gdeltproject.org/gdeltv2/masterfilelist.txt`. Each entry is a 15-minute compressed CSV of global events. We fetch one file per day (the midnight UTC snapshot) for the lookback window, filter for relevant geopolitical actors, and cache the result.

**GDELT Events columns used:**
- `QuadClass` — event category (1–4)
- `GoldsteinScale` — stability impact score (−10 to +10)
- `Actor1CountryCode` / `Actor2CountryCode` — ISO country codes
- `NumMentions` — article coverage weight

In [ ]:
GDELT_COLS = [
    'GLOBALEVENTID', 'SQLDATE', 'Actor1CountryCode', 'Actor2CountryCode',
    'EventCode', 'EventRootCode', 'QuadClass', 'GoldsteinScale',
    'NumMentions', 'NumSources', 'NumArticles', 'AvgTone'
]

# Column positions in GDELT 2.0 (0-indexed)
COL_IDX = {
    'GLOBALEVENTID':      0,
    'SQLDATE':            1,
    'Actor1CountryCode':  7,
    'Actor2CountryCode': 17,
    'EventCode':         26,
    'EventRootCode':     28,
    'QuadClass':         29,
    'GoldsteinScale':    30,
    'NumMentions':       31,
    'NumSources':        32,
    'NumArticles':       33,
    'AvgTone':           34,
}
USE_COLS = list(COL_IDX.values())


def fetch_gdelt_day(date_str: str, sleep: float = 0.3) -> pd.DataFrame:
    """Download GDELT events for one calendar day (YYYYMMDD)."""
    url = f'http://data.gdeltproject.org/gdeltv2/{date_str}000000.export.CSV.zip'
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
    except Exception:
        return pd.DataFrame()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        fname = z.namelist()[0]
        with z.open(fname) as f:
            df = pd.read_csv(
                f, sep='\t', header=None,
                usecols=USE_COLS,
                names=GDELT_COLS,
                on_bad_lines='skip',
                dtype=str
            )

    time.sleep(sleep)
    return df


def is_relevant(row) -> bool:
    a1 = str(row['Actor1CountryCode']).upper()
    a2 = str(row['Actor2CountryCode']).upper()
    return a1 in CONFLICT_ACTORS or a2 in CONFLICT_ACTORS


def load_gdelt(lookback: int = LOOKBACK_DAYS) -> pd.DataFrame:
    if os.path.exists(CACHE_FILE):
        print(f'Loading cached GDELT data from {CACHE_FILE}')
        df = pd.read_csv(CACHE_FILE)
        df['Date'] = pd.to_datetime(df['Date']).dt.date
        print(f'  {len(df):,} events loaded.')
        return df

    print(f'Fetching {lookback} days of GDELT data — this will take a few minutes.')
    end   = date.today() - timedelta(days=1)   # yesterday (today may be incomplete)
    start = end - timedelta(days=lookback)
    all_dates = pd.date_range(start, end, freq='B')  # business days only

    frames = []
    for i, d in enumerate(all_dates, 1):
        date_str = d.strftime('%Y%m%d')
        day_df = fetch_gdelt_day(date_str)
        if day_df.empty:
            continue
        day_df = day_df[day_df.apply(is_relevant, axis=1)].copy()
        day_df['Date'] = d.date()
        frames.append(day_df)
        if i % 20 == 0:
            print(f'  [{i}/{len(all_dates)}] {date_str} — running total: {sum(len(f) for f in frames):,}')

    full = pd.concat(frames, ignore_index=True)
    full.to_csv(CACHE_FILE, index=False)
    print(f'Saved {len(full):,} events to {CACHE_FILE}')
    return full


raw = load_gdelt()
print(raw.head())

## 3. Daily geopolitical signal construction

We aggregate raw events into daily conflict/cooperation scores:

- **Conflict_Count** — events with QuadClass ∈ {3, 4} weighted by NumMentions
- **Coop_Count** — events with QuadClass ∈ {1, 2} weighted by NumMentions  
- **Goldstein_Mean** — average Goldstein scale (negative = more conflict)
- **Geo_LogRatio** — log((Conflict_Count + 1) / (Coop_Count + 1))
- **Net_Conflict** — Conflict_Count − Coop_Count

In [ ]:
df = raw.copy()
df['QuadClass']      = pd.to_numeric(df['QuadClass'],      errors='coerce')
df['GoldsteinScale'] = pd.to_numeric(df['GoldsteinScale'], errors='coerce')
df['NumMentions']    = pd.to_numeric(df['NumMentions'],    errors='coerce').fillna(1)
df = df.dropna(subset=['QuadClass'])

df['IsConflict'] = df['QuadClass'].isin(CONFLICT_CLASSES).astype(int)
df['IsCoop']     = df['QuadClass'].isin(COOPERATION_CLASSES).astype(int)

daily = df.groupby('Date').agg(
    Conflict_Count  = ('IsConflict',    lambda x: (x * df.loc[x.index, 'NumMentions']).sum()),
    Coop_Count      = ('IsCoop',        lambda x: (x * df.loc[x.index, 'NumMentions']).sum()),
    Goldstein_Mean  = ('GoldsteinScale', 'mean'),
    Total_Events    = ('GLOBALEVENTID', 'count'),
).reset_index()

# Fill calendar gaps (weekends, holidays) with zero
full_idx = pd.date_range(daily['Date'].min(), daily['Date'].max(), freq='D')
daily = daily.set_index('Date').reindex(full_idx.date).fillna(0).reset_index()
daily.rename(columns={'index': 'Date'}, inplace=True)

daily['Geo_LogRatio'] = np.log(
    (daily['Conflict_Count'] + 1) / (daily['Coop_Count'] + 1)
)
daily['Net_Conflict'] = daily['Conflict_Count'] - daily['Coop_Count']

print('Daily geopolitical signals:')
print(daily.tail(10).to_string(index=False))

## 4. Market data

We fetch daily closing prices for the iShares U.S. Aerospace & Defence ETF (`ITA`), the S&P 500 ETF (`SPY`), and individual defence names. `ITA` provides direct exposure to defence equities without the noise of diversified conglomerates.

In [ ]:
start_dt = (date.today() - timedelta(days=LOOKBACK_DAYS + 30)).strftime('%Y-%m-%d')
end_dt   = date.today().strftime('%Y-%m-%d')

tickers = [DEFENCE_TICKER, MARKET_TICKER] + DEFENCE_PEERS
raw_prices = yf.download(tickers, start=start_dt, end=end_dt, auto_adjust=True, progress=False)

prices = raw_prices['Close'].copy()
prices.index = pd.to_datetime(prices.index).date
prices = prices.dropna(how='any')

returns = prices.pct_change().dropna()
returns['Excess_ITA'] = returns[DEFENCE_TICKER] - returns[MARKET_TICKER]

target = returns[['Excess_ITA']].copy()
target['Date'] = target.index
target = target.reset_index(drop=True)

print(f'Price data: {prices.shape[0]} trading days')
print(target.tail())

## 5. Merge and diagnostic plots

In [ ]:
daily['Date'] = pd.to_datetime(daily['Date']).dt.date
model_df = pd.merge(target, daily, on='Date', how='inner').sort_values('Date').reset_index(drop=True)

print(f'Merged dataset: {len(model_df)} rows')
print(model_df.head(5).to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(model_df['Date'], model_df['Conflict_Count'], label='Conflict events', color='crimson', alpha=0.8)
axes[0].plot(model_df['Date'], model_df['Coop_Count'],     label='Cooperation events', color='steelblue', alpha=0.8)
axes[0].set_title('Daily conflict vs cooperation event volume (GDELT)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(model_df['Date'], model_df['Excess_ITA'], color='darkorange', alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title('Daily ITA excess return vs SPY')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Rolling Z-scores and spike detection

A 30-day rolling baseline normalises geopolitical intensity so that spike detection is relative to recent history rather than the full-sample mean — important because baseline conflict levels drift over time.

In [ ]:
W = ROLLING_WINDOW

for col in ['Conflict_Count', 'Coop_Count']:
    model_df[f'{col}_ZRoll'] = (
        (model_df[col] - model_df[col].rolling(W).mean()) /
         model_df[col].rolling(W).std()
    )

model_df['Esc_Spike_15']   = (model_df['Conflict_Count_ZRoll'] > 1.5).astype(int)
model_df['Esc_Spike_20']   = (model_df['Conflict_Count_ZRoll'] > 2.0).astype(int)
model_df['DeEsc_Spike_15'] = (model_df['Coop_Count_ZRoll']    > 1.5).astype(int)
model_df['DeEsc_Spike_20'] = (model_df['Coop_Count_ZRoll']    > 2.0).astype(int)

esc_15  = model_df[model_df['Esc_Spike_15']   == 1]['Date'].tolist()
esc_20  = model_df[model_df['Esc_Spike_20']   == 1]['Date'].tolist()
deesc_15= model_df[model_df['DeEsc_Spike_15'] == 1]['Date'].tolist()
deesc_20= model_df[model_df['DeEsc_Spike_20'] == 1]['Date'].tolist()

print(f'Escalation spikes   Z>1.5: {len(esc_15)}   Z>2.0: {len(esc_20)}')
print(f'De-escalation spikes Z>1.5: {len(deesc_15)}  Z>2.0: {len(deesc_20)}')

plt.figure(figsize=(12, 5))
plt.plot(model_df['Date'], model_df['Conflict_Count_ZRoll'], label='Conflict Z-score', color='crimson', alpha=0.8)
plt.axhline(1.5, linestyle='--', color='crimson', linewidth=0.8, alpha=0.6)
plt.axhline(2.0, linestyle='-.',  color='darkred',  linewidth=0.8, alpha=0.6)
plt.scatter(
    [d for d in esc_20 if d in model_df['Date'].values],
    model_df[model_df['Date'].isin(esc_20)]['Conflict_Count_ZRoll'],
    color='darkred', s=60, zorder=5, label='Spike days (Z>2.0)'
)
plt.title('Conflict intensity rolling Z-score with spike detection')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Event study — ITA excess return on spike days

In [ ]:
def event_stats(spike_dates, label):
    subset = model_df[model_df['Date'].isin(spike_dates)]['Excess_ITA']
    return {'Label': label, 'N': len(subset), 'Mean': subset.mean(), 'Std': subset.std()}

rows = [
    event_stats(esc_15,   'Escalation Z>1.5'),
    event_stats(esc_20,   'Escalation Z>2.0'),
    event_stats(deesc_15, 'De-escalation Z>1.5'),
    event_stats(deesc_20, 'De-escalation Z>2.0'),
]
summary = pd.DataFrame(rows)
print('=== EVENT STUDY RESULTS ===')
print(summary.to_string(index=False))

plt.figure(figsize=(9, 5))
colours = ['green', 'darkgreen', 'red', 'darkred']
plt.bar(summary['Label'], summary['Mean'], color=colours)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('ITA average same-day excess return on geopolitical spike days')
plt.ylabel('Average excess return')
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Peer event study — individual defence stocks

In [ ]:
returns_reset = returns.copy()
returns_reset['Date'] = returns_reset.index

def peer_reaction(spike_dates, label):
    td = [d for d in spike_dates if d in returns_reset.index]
    row = {'Threshold': label, 'N': len(td)}
    for t in DEFENCE_PEERS:
        row[t] = returns_reset.loc[td, t].mean()
    row['ITA_Excess'] = returns_reset.loc[td, 'Excess_ITA'].mean()
    return row

peer_df = pd.DataFrame([
    peer_reaction(esc_15, 'Z>1.5'),
    peer_reaction(esc_20, 'Z>2.0'),
])

print('=== PEER ESCALATION REACTION ===')
print(peer_df.to_string(index=False))

## 9. Market-model abnormal returns

For each peer stock, we estimate a rolling 120-day OLS market model (stock ~ alpha + beta × SPY) and compute daily abnormal returns as the residual. This isolates stock-specific reactions from broad market moves.

In [ ]:
ESTIMATION_WINDOW = 120
mkt = returns[MARKET_TICKER].values
abnormal = pd.DataFrame(index=returns.index)

for ticker in DEFENCE_PEERS:
    y_all = pd.to_numeric(returns[ticker], errors='coerce')
    alphas, betas = [], []

    for i in range(len(returns)):
        if i < ESTIMATION_WINDOW:
            alphas.append(np.nan); betas.append(np.nan); continue
        y_w = y_all.iloc[i-ESTIMATION_WINDOW:i].values.astype(float)
        x_w = mkt[i-ESTIMATION_WINDOW:i].astype(float)
        valid = ~(np.isnan(y_w) | np.isnan(x_w))
        if valid.sum() < 30:
            alphas.append(np.nan); betas.append(np.nan); continue
        X = np.column_stack([np.ones(valid.sum()), x_w[valid]])
        b = np.linalg.lstsq(X, y_w[valid], rcond=None)[0]
        alphas.append(b[0]); betas.append(b[1])

    alpha_s = pd.Series(alphas, index=returns.index)
    beta_s  = pd.Series(betas,  index=returns.index)
    expected = alpha_s + beta_s * returns[MARKET_TICKER]
    abnormal[ticker] = y_all - expected

print('Abnormal returns computed.')
print(abnormal.dropna().tail())

## 10. Bootstrap significance test

In [ ]:
N_BOOT = 5000
results = {}

trading_spikes = [d for d in esc_20 if d in abnormal.index]

for ticker in DEFENCE_PEERS:
    obs_vals = abnormal.loc[abnormal.index.isin(esc_20), ticker].dropna()
    if len(obs_vals) == 0:
        continue
    obs_mean = obs_vals.mean()
    full     = abnormal[ticker].dropna().values
    boot_means = np.array([
        np.random.choice(full, size=len(obs_vals), replace=False).mean()
        for _ in range(N_BOOT)
    ])
    pctile = (boot_means < obs_mean).mean()
    results[ticker] = {'Observed_AR': obs_mean, 'Bootstrap_Percentile': pctile}

boot_df = pd.DataFrame(results).T.sort_values('Observed_AR', ascending=False)
print('=== BOOTSTRAP SIGNIFICANCE (Z>2.0 escalation days) ===')
print(boot_df)

## 11. Regression robustness check

In [ ]:
reg = model_df.dropna().copy()
y = reg['Excess_ITA'].values.astype(float)
x = reg['Geo_LogRatio'].values.astype(float)

X1 = np.column_stack([np.ones(len(x)), x])
b1 = np.linalg.lstsq(X1, y, rcond=None)[0]
y_hat = X1 @ b1
r2 = 1 - np.sum((y - y_hat)**2) / np.sum((y - y.mean())**2)

print(f'Univariate: Excess_ITA ~ Geo_LogRatio')
print(f'  Intercept: {b1[0]:.6f}')
print(f'  Slope:     {b1[1]:.6f}')
print(f'  R²:        {r2:.4f}')

features = ['Geo_LogRatio', 'Net_Conflict', 'Total_Events']
X2 = np.column_stack([np.ones(len(reg))] + [reg[f].astype(float).values for f in features])
b2 = np.linalg.lstsq(X2, y, rcond=None)[0]
y_hat2 = X2 @ b2
r2_m = 1 - np.sum((y - y_hat2)**2) / np.sum((y - y.mean())**2)
print(f'\nMultivariate R²: {r2_m:.4f}')

plt.figure(figsize=(9, 5))
plt.scatter(reg['Geo_LogRatio'], reg['Excess_ITA'], alpha=0.5, s=15, label='Observed')
order = reg['Geo_LogRatio'].argsort().values
plt.plot(reg['Geo_LogRatio'].values[order], y_hat[order], color='crimson', lw=2, label='Fitted')
plt.xlabel('Geo_LogRatio (log conflict/cooperation)')
plt.ylabel('ITA excess return')
plt.title('Geopolitical polarity vs defence sector excess return')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()